# SagingSense: A Convolutional Neural Network-Based Banana Ripeness Classification System for Fruit Quality Assessment

This notebook is Version 2 of the SagingSense model development workflow.

The goal is to build and compare multiple image classification models for banana ripeness classification.

## Models Included

1. Baseline CNN
2. Improved CNN
3. MobileNetV2 Transfer Learning
4. EfficientNetB0 Transfer Learning

## Evaluation Metrics

The notebook will report:

- Accuracy
- Precision
- Recall
- F1-score
- Confusion matrix
- Correct and incorrect prediction count per class
- Best model summary

## Acronyms

| Acronym | Meaning |
|---|---|
| CNN | Convolutional Neural Network |
| RGB | Red, Green, Blue |
| GPU | Graphics Processing Unit |
| F1 | Harmonic mean of precision and recall |
| API | Application Programming Interface |

## Import Libraries

In [1]:
import os
import json
import random
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)


TensorFlow version: 2.18.0
GPU devices: []


## Dataset Path Configuration
Change the path depending on where your dataset is located.

In [2]:
# ============================================================
# DATASET PATH CONFIGURATION
# ============================================================
from pathlib import Path

DATASET_NAME = "banana-ripeness-classification-dataset"
candidates = []

# Search from current directory upward so this works regardless of where Jupyter was launched.
for base in [Path.cwd(), *Path.cwd().parents]:
    candidates.append(base / "kaggle" / "input" / DATASET_NAME)

# Kaggle notebook absolute path fallback.
candidates.append(Path("/kaggle/input") / DATASET_NAME)

DATASET_ROOT = next((c for c in candidates if c.exists()), None)

if DATASET_ROOT is None:
    checked = "\n".join(str(c) for c in candidates)
    raise FileNotFoundError(
        "Could not locate dataset folder. Checked:\n"
        f"{checked}\n"
        f"Current working directory: {Path.cwd()}"
    )

print("Dataset root:", DATASET_ROOT)

Dataset root: E:\GitHub\SagingSense\kaggle\input\banana-ripeness-classification-dataset


## Automatically Detect Image Folder

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def count_images_in_class_folders(folder):
    folder = Path(folder)
    class_counts = {}
    for subfolder in sorted(folder.iterdir()):
        if subfolder.is_dir():
            image_count = sum(
                1 for file in subfolder.rglob("*")
                if file.suffix.lower() in IMAGE_EXTENSIONS
            )
            if image_count > 0:
                class_counts[subfolder.name] = image_count
    return class_counts

required_splits = ["train", "valid", "test"]
split_dirs = {split: DATASET_ROOT / split for split in required_splits}

missing = [split for split, path in split_dirs.items() if not path.exists()]
if missing:
    raise FileNotFoundError(
        f"Missing required split folder(s): {missing}\n"
        f"Expected under: {DATASET_ROOT}"
    )

train_class_counts = count_images_in_class_folders(split_dirs["train"])
if len(train_class_counts) < 2:
    raise ValueError(
        f"Train folder does not contain enough class subfolders with images: {split_dirs['train']}"
    )

class_names = sorted(train_class_counts.keys())
num_classes = len(class_names)

# Use ripeness classes from train folder (not split names).
DATA_DIR = split_dirs["train"]
class_counts = train_class_counts
total_images = sum(class_counts.values())

# Optional consistency check: valid/test should have the same class folders.
for split in ["valid", "test"]:
    split_classes = set(count_images_in_class_folders(split_dirs[split]).keys())
    if split_classes != set(class_names):
        raise ValueError(
            "Class folder mismatch across splits.\n"
            f"train={class_names}\n"
            f"{split}={sorted(split_classes)}"
        )

print("Using training folder for class discovery:")
print(DATA_DIR)
print("\nDetected ripeness classes:")
for class_name in class_names:
    print(f"{class_name}: {class_counts[class_name]} images")
print("\nTotal training images:", total_images)
print("Number of classes:", num_classes)


## Dataset Summary Table

In [ ]:
dataset_summary = pd.DataFrame({
    "Class": class_names,
    "Image Count": [class_counts[class_name] for class_name in class_names]
})

dataset_summary["Percentage"] = (
    dataset_summary["Image Count"] / dataset_summary["Image Count"].sum() * 100
).round(2)

print("Dataset Summary")
print(dataset_summary.to_string(index=False))

plt.figure(figsize=(10, 5))
plt.bar(dataset_summary["Class"], dataset_summary["Image Count"])
plt.title("Dataset Image Count per Banana Ripeness Class")
plt.xlabel("Banana Ripeness Class")
plt.ylabel("Number of Images")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## Build Image Path Table

In [ ]:
class_to_index = {class_name: index for index, class_name in enumerate(class_names)}
index_to_class = {index: class_name for class_name, index in class_to_index.items()}

def collect_split_data(split_name):
    split_path = split_dirs[split_name]
    split_paths = []
    split_labels = []
    for class_name in class_names:
        class_folder = split_path / class_name
        for file in class_folder.rglob("*"):
            if file.suffix.lower() in IMAGE_EXTENSIONS:
                split_paths.append(str(file))
                split_labels.append(class_to_index[class_name])
    return np.array(split_paths), np.array(split_labels)

# Use the dataset's provided split folders directly.
X_train, y_train = collect_split_data("train")
X_val, y_val = collect_split_data("valid")
X_test, y_test = collect_split_data("test")

# Combined arrays are kept for EDA/visualization compatibility.
image_paths = np.concatenate([X_train, X_val, X_test])
labels = np.concatenate([y_train, y_val, y_test])
splits = np.concatenate([
    np.array(["train"] * len(X_train)),
    np.array(["valid"] * len(X_val)),
    np.array(["test"] * len(X_test))
])

image_table = pd.DataFrame({
    "image_path": image_paths,
    "label_index": labels,
    "split": splits
})

image_table["label_name"] = image_table["label_index"].map(index_to_class)

print("Loaded images by original dataset split:")
print("Training images:", len(X_train))
print("Validation images:", len(X_val))
print("Testing images:", len(X_test))
print("Total images loaded:", len(image_table))
print("\nFirst 10 image records:")
print(image_table.head(10).to_string(index=False))


## Display Sample Images

In [ ]:
plt.figure(figsize=(12, 8))

sample_indices = random.sample(range(len(image_paths)), min(12, len(image_paths)))

for plot_index, image_index in enumerate(sample_indices):
    img = keras.utils.load_img(image_paths[image_index], target_size=(160, 160))
    label_name = index_to_class[int(labels[image_index])]

    plt.subplot(3, 4, plot_index + 1)
    plt.imshow(img)
    plt.title(label_name)
    plt.axis("off")

plt.suptitle("Sample Banana Images from the Dataset")
plt.tight_layout()
plt.show()

## Train, Validation, and Test Split

We will use this split:

70% training,15% validation,15% testing

In [ ]:
# Keep the original dataset split (no random re-splitting).
print("Training images:", len(X_train))
print("Validation images:", len(X_val))
print("Testing images:", len(X_test))

split_summary = pd.DataFrame({
    "Split": ["Training", "Validation", "Testing"],
    "Image Count": [len(X_train), len(X_val), len(X_test)]
})

split_summary["Percentage"] = (
    split_summary["Image Count"] / split_summary["Image Count"].sum() * 100
).round(2)

print("\nSplit Summary:")
print(split_summary.to_string(index=False))


## Class Distribution per Split

In [ ]:
def make_split_distribution_table(y_values, split_name):
    temp_df = pd.DataFrame({
        "label_index": y_values
    })

    temp_df["Class"] = temp_df["label_index"].map(index_to_class)

    counts = temp_df["Class"].value_counts().reindex(class_names, fill_value=0)

    output = pd.DataFrame({
        "Class": counts.index,
        split_name: counts.values
    })

    return output


train_dist = make_split_distribution_table(y_train, "Training")
val_dist = make_split_distribution_table(y_val, "Validation")
test_dist = make_split_distribution_table(y_test, "Testing")

distribution_table = train_dist.merge(val_dist, on="Class").merge(test_dist, on="Class")
distribution_table["Total"] = (
    distribution_table["Training"] +
    distribution_table["Validation"] +
    distribution_table["Testing"]
)

print("Class Distribution per Split:")
print(distribution_table.to_string(index=False))

## Create TensorFlow Datasets
For this version, the image tensors are kept in the range: 0 to 255

Each model will apply its own preprocessing internally.

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def load_image_raw(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = tf.cast(image, tf.float32)
    return image, label


def create_tf_dataset(paths, labels, shuffle=False):
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    dataset = dataset.map(load_image_raw, num_parallel_calls=AUTOTUNE)

    if shuffle:
        dataset = dataset.shuffle(
            buffer_size=len(paths),
            seed=RANDOM_SEED,
            reshuffle_each_iteration=True
        )

    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(AUTOTUNE)

    return dataset


train_ds = create_tf_dataset(X_train, y_train, shuffle=True)
val_ds = create_tf_dataset(X_val, y_val, shuffle=False)
test_ds = create_tf_dataset(X_test, y_test, shuffle=False)

print("Training dataset:", train_ds)
print("Validation dataset:", val_ds)
print("Testing dataset:", test_ds)

## Shared Data Augmentation Layer

In [ ]:
data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.08),
        layers.RandomZoom(0.10),
        layers.RandomContrast(0.10),
    ],
    name="data_augmentation"
)

--------
## Part 1: Model Builders
-----
#### Baseline CNN Model

In [ ]:
def build_baseline_cnn():
    model = keras.Sequential(
        [
            layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
            layers.Rescaling(1.0 / 255.0),

            layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
            layers.MaxPooling2D(),

            layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
            layers.MaxPooling2D(),

            layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
            layers.MaxPooling2D(),

            layers.Flatten(),
            layers.Dense(128, activation="relu"),
            layers.Dense(num_classes, activation="softmax")
        ],
        name="Baseline_CNN"
    )

    return model

#### Improved CNN Model

In [ ]:
def build_improved_cnn():
    model = keras.Sequential(
        [
            layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
            data_augmentation,
            layers.Rescaling(1.0 / 255.0),

            layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
            layers.BatchNormalization(),
            layers.MaxPooling2D(),

            layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
            layers.BatchNormalization(),
            layers.MaxPooling2D(),

            layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
            layers.BatchNormalization(),
            layers.MaxPooling2D(),

            layers.Conv2D(256, (3, 3), activation="relu", padding="same"),
            layers.BatchNormalization(),
            layers.MaxPooling2D(),

            layers.GlobalAveragePooling2D(),

            layers.Dense(256, activation="relu"),
            layers.Dropout(0.40),

            layers.Dense(num_classes, activation="softmax")
        ],
        name="Improved_CNN"
    )

    return model

### MobileNetV2 Transfer Learning Model

In [ ]:
def build_mobilenetv2_model():
    try:
        base_model = keras.applications.MobileNetV2(
            input_shape=(IMG_SIZE, IMG_SIZE, 3),
            include_top=False,
            weights="imagenet"
        )
        print("MobileNetV2 loaded with ImageNet weights.")
    except Exception as error:
        print("Could not load ImageNet weights for MobileNetV2.")
        print("Reason:", error)
        print("Using MobileNetV2 with random weights instead.")
        base_model = keras.applications.MobileNetV2(
            input_shape=(IMG_SIZE, IMG_SIZE, 3),
            include_top=False,
            weights=None
        )

    base_model.trainable = False

    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = data_augmentation(inputs)
    x = keras.applications.mobilenet_v2.preprocess_input(x)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(128, activation="relu")(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inputs, outputs, name="MobileNetV2_Transfer_Learning")

    return model

### EfficientNetB0 Transfer Learning Model

In [ ]:
def build_efficientnetb0_model():
    try:
        base_model = keras.applications.EfficientNetB0(
            input_shape=(IMG_SIZE, IMG_SIZE, 3),
            include_top=False,
            weights="imagenet"
        )
        print("EfficientNetB0 loaded with ImageNet weights.")
    except Exception as error:
        print("Could not load ImageNet weights for EfficientNetB0.")
        print("Reason:", error)
        print("Using EfficientNetB0 with random weights instead.")
        base_model = keras.applications.EfficientNetB0(
            input_shape=(IMG_SIZE, IMG_SIZE, 3),
            include_top=False,
            weights=None
        )

    base_model.trainable = False

    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = data_augmentation(inputs)

    # EfficientNet in TensorFlow Keras includes preprocessing internally.
    x = base_model(x, training=False)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(128, activation="relu")(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inputs, outputs, name="EfficientNetB0_Transfer_Learning")

    return model

----------
## Part 2: Training and Evaluation Utilities
------------
### Compile Function

In [ ]:
def compile_model(model, learning_rate=0.001):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

### Training Function

In [ ]:
def train_model(model, model_name, epochs=15, learning_rate=0.001):
    print("=" * 80)
    print(f"Training model: {model_name}")
    print("=" * 80)

    model = compile_model(model, learning_rate=learning_rate)

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=4,
            restore_best_weights=True
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.3,
            patience=2,
            min_lr=1e-6
        )
    ]

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        callbacks=callbacks,
        verbose=1
    )

    return model, history

## Prediction and Evaluation Function

In [ ]:
def evaluate_model(model, model_name):
    print("=" * 80)
    print(f"Evaluating model: {model_name}")
    print("=" * 80)

    test_loss, test_accuracy = model.evaluate(test_ds, verbose=1)

    y_true = []
    y_pred = []
    y_confidence = []

    for images, labels_batch in test_ds:
        predictions = model.predict(images, verbose=0)

        batch_pred = np.argmax(predictions, axis=1)
        batch_confidence = np.max(predictions, axis=1)

        y_true.extend(labels_batch.numpy())
        y_pred.extend(batch_pred)
        y_confidence.extend(batch_confidence)

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_confidence = np.array(y_confidence)

    report_dict = classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )

    report_df = pd.DataFrame(report_dict).transpose()

    weighted_precision = report_dict["weighted avg"]["precision"]
    weighted_recall = report_dict["weighted avg"]["recall"]
    weighted_f1 = report_dict["weighted avg"]["f1-score"]

    cm = confusion_matrix(y_true, y_pred)

    results_df = pd.DataFrame({
        "Image Path": X_test,
        "True Label Index": y_true,
        "Predicted Label Index": y_pred,
        "Confidence": y_confidence
    })

    results_df["True Label"] = results_df["True Label Index"].map(index_to_class)
    results_df["Predicted Label"] = results_df["Predicted Label Index"].map(index_to_class)
    results_df["Correct"] = results_df["True Label Index"] == results_df["Predicted Label Index"]

    summary = {
        "Model": model_name,
        "Test Loss": test_loss,
        "Test Accuracy": test_accuracy,
        "Weighted Precision": weighted_precision,
        "Weighted Recall": weighted_recall,
        "Weighted F1-Score": weighted_f1,
        "Correct Predictions": int(results_df["Correct"].sum()),
        "Wrong Predictions": int((~results_df["Correct"]).sum()),
        "Total Test Images": int(len(results_df))
    }

    return summary, report_df, cm, results_df

## Plot Training Curves

In [ ]:
def plot_training_curves(history, model_name):
    history_df = pd.DataFrame(history.history)

    print(f"Training history for {model_name}:")
    print(history_df.round(4).to_string(index=False))

    plt.figure(figsize=(8, 5))
    plt.plot(history_df["accuracy"], label="Training Accuracy")
    plt.plot(history_df["val_accuracy"], label="Validation Accuracy")
    plt.title(f"{model_name} Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(history_df["loss"], label="Training Loss")
    plt.plot(history_df["val_loss"], label="Validation Loss")
    plt.title(f"{model_name} Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

## Plot Confusion Matrix

In [ ]:
def plot_confusion_matrix(cm, model_name):
    cm_df = pd.DataFrame(
        cm,
        index=[f"True: {name}" for name in class_names],
        columns=[f"Predicted: {name}" for name in class_names]
    )

    print(f"Confusion Matrix for {model_name}:")
    print(cm_df.to_string())

    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation="nearest")
    plt.title(f"{model_name} Confusion Matrix")
    plt.colorbar()

    tick_marks = np.arange(len(class_names))
    plt.xticks(tick_marks, class_names, rotation=45, ha="right")
    plt.yticks(tick_marks, class_names)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(
                j,
                i,
                cm[i, j],
                horizontalalignment="center",
                verticalalignment="center"
            )

    plt.ylabel("True Label")
    plt.xlabel("Predicted Label")
    plt.tight_layout()
    plt.show()

## Correct and Wrong Count per Class

In [ ]:
def show_correct_wrong_by_class(results_df, model_name):
    print(f"Correct and Wrong Prediction Count per Class for {model_name}")

    table = results_df.groupby("True Label")["Correct"].value_counts().unstack(fill_value=0)

    if True not in table.columns:
        table[True] = 0

    if False not in table.columns:
        table[False] = 0

    table = table.rename(columns={
        True: "Correct",
        False: "Wrong"
    })

    table["Total"] = table["Correct"] + table["Wrong"]
    table["Accuracy per Class"] = (table["Correct"] / table["Total"]).round(4)

    print(table.to_string())

    return table

## Show Wrong Predictions

In [ ]:
def show_wrong_predictions(results_df, model_name, max_images=12):
    wrong_predictions = results_df[results_df["Correct"] == False].copy()
    wrong_predictions = wrong_predictions.sort_values("Confidence", ascending=False)

    print(f"Number of wrong predictions for {model_name}:", len(wrong_predictions))

    if len(wrong_predictions) == 0:
        print("No wrong predictions found.")
        return

    sample_wrong = wrong_predictions.head(max_images)

    plt.figure(figsize=(12, 8))

    for plot_index, (_, row) in enumerate(sample_wrong.iterrows()):
        img = keras.utils.load_img(row["Image Path"], target_size=(160, 160))

        plt.subplot(3, 4, plot_index + 1)
        plt.imshow(img)
        plt.title(
            f"True: {row['True Label']}\n"
            f"Pred: {row['Predicted Label']}\n"
            f"Conf: {row['Confidence']:.2f}"
        )
        plt.axis("off")

    plt.suptitle(f"Sample Wrong Predictions: {model_name}")
    plt.tight_layout()
    plt.show()

----------------
## Part 3: Run Model Experiments
-------------
You may run all models, or turn some off if training takes too long.

### Experiment Settings

In [ ]:
RUN_BASELINE_CNN = True
RUN_IMPROVED_CNN = True
RUN_MOBILENETV2 = True
RUN_EFFICIENTNETB0 = True

EPOCHS_BASELINE = 12
EPOCHS_IMPROVED = 20
EPOCHS_TRANSFER = 10

all_summaries = []
all_reports = {}
all_confusion_matrices = {}
all_results = {}
trained_models = {}
training_histories = {}

## Run Baseline CNN

In [ ]:
if RUN_BASELINE_CNN:
    tf.keras.backend.clear_session()

    model_name = "Baseline CNN"
    baseline_model = build_baseline_cnn()
    baseline_model.summary()

    baseline_model, baseline_history = train_model(
        baseline_model,
        model_name=model_name,
        epochs=EPOCHS_BASELINE,
        learning_rate=0.001
    )

    summary, report_df, cm, results_df = evaluate_model(baseline_model, model_name)

    all_summaries.append(summary)
    all_reports[model_name] = report_df
    all_confusion_matrices[model_name] = cm
    all_results[model_name] = results_df
    trained_models[model_name] = baseline_model
    training_histories[model_name] = baseline_history

    plot_training_curves(baseline_history, model_name)
    plot_confusion_matrix(cm, model_name)
    show_correct_wrong_by_class(results_df, model_name)
    show_wrong_predictions(results_df, model_name)

## Run Improved CNN

In [ ]:
if RUN_IMPROVED_CNN:
    tf.keras.backend.clear_session()

    model_name = "Improved CNN"
    improved_model = build_improved_cnn()
    improved_model.summary()

    improved_model, improved_history = train_model(
        improved_model,
        model_name=model_name,
        epochs=EPOCHS_IMPROVED,
        learning_rate=0.001
    )

    summary, report_df, cm, results_df = evaluate_model(improved_model, model_name)

    all_summaries.append(summary)
    all_reports[model_name] = report_df
    all_confusion_matrices[model_name] = cm
    all_results[model_name] = results_df
    trained_models[model_name] = improved_model
    training_histories[model_name] = improved_history

    plot_training_curves(improved_history, model_name)
    plot_confusion_matrix(cm, model_name)
    show_correct_wrong_by_class(results_df, model_name)
    show_wrong_predictions(results_df, model_name)

### Run MobileNetV2 Transfer Learning

In [ ]:
if RUN_MOBILENETV2:
    tf.keras.backend.clear_session()

    model_name = "MobileNetV2 Transfer Learning"
    mobilenet_model = build_mobilenetv2_model()
    mobilenet_model.summary()

    mobilenet_model, mobilenet_history = train_model(
        mobilenet_model,
        model_name=model_name,
        epochs=EPOCHS_TRANSFER,
        learning_rate=0.0005
    )

    summary, report_df, cm, results_df = evaluate_model(mobilenet_model, model_name)

    all_summaries.append(summary)
    all_reports[model_name] = report_df
    all_confusion_matrices[model_name] = cm
    all_results[model_name] = results_df
    trained_models[model_name] = mobilenet_model
    training_histories[model_name] = mobilenet_history

    plot_training_curves(mobilenet_history, model_name)
    plot_confusion_matrix(cm, model_name)
    show_correct_wrong_by_class(results_df, model_name)
    show_wrong_predictions(results_df, model_name)

## Run EfficientNetB0 Transfer Learning

In [ ]:
if RUN_EFFICIENTNETB0:
    tf.keras.backend.clear_session()

    model_name = "EfficientNetB0 Transfer Learning"
    efficientnet_model = build_efficientnetb0_model()
    efficientnet_model.summary()

    efficientnet_model, efficientnet_history = train_model(
        efficientnet_model,
        model_name=model_name,
        epochs=EPOCHS_TRANSFER,
        learning_rate=0.0005
    )

    summary, report_df, cm, results_df = evaluate_model(efficientnet_model, model_name)

    all_summaries.append(summary)
    all_reports[model_name] = report_df
    all_confusion_matrices[model_name] = cm
    all_results[model_name] = results_df
    trained_models[model_name] = efficientnet_model
    training_histories[model_name] = efficientnet_history

    plot_training_curves(efficientnet_history, model_name)
    plot_confusion_matrix(cm, model_name)
    show_correct_wrong_by_class(results_df, model_name)
    show_wrong_predictions(results_df, model_name)

-----------
## Part 4: Model Comparison
---------
#### Model Comparison Table

In [ ]:
comparison_df = pd.DataFrame(all_summaries)

if len(comparison_df) == 0:
    raise ValueError("No model experiments were run. Please set at least one RUN option to True.")

comparison_df = comparison_df.sort_values("Weighted F1-Score", ascending=False).reset_index(drop=True)

numeric_columns = [
    "Test Loss",
    "Test Accuracy",
    "Weighted Precision",
    "Weighted Recall",
    "Weighted F1-Score"
]

comparison_df[numeric_columns] = comparison_df[numeric_columns].round(4)

print("Model Comparison Table:")
print(comparison_df.to_string(index=False))

best_model_name = comparison_df.loc[0, "Model"]
best_model = trained_models[best_model_name]

print("\nBest Model Based on Weighted F1-Score:")
print(best_model_name)

## Visualize Model Comparison

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(comparison_df["Model"], comparison_df["Test Accuracy"])
plt.title("Model Comparison by Test Accuracy")
plt.xlabel("Model")
plt.ylabel("Test Accuracy")
plt.xticks(rotation=30, ha="right")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
plt.bar(comparison_df["Model"], comparison_df["Weighted F1-Score"])
plt.title("Model Comparison by Weighted F1-Score")
plt.xlabel("Model")
plt.ylabel("Weighted F1-Score")
plt.xticks(rotation=30, ha="right")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

## Best Model Classification Report

In [ ]:
print("Best Model:", best_model_name)

best_report = all_reports[best_model_name]

print("\nClassification Report for Best Model:")
print(best_report.round(4).to_string())

## Best Model Confusion Matrix

In [ ]:
best_cm = all_confusion_matrices[best_model_name]
plot_confusion_matrix(best_cm, best_model_name)

## Best Model Correct and Wrong Prediction Table

In [ ]:
best_results_df = all_results[best_model_name]

best_correct_wrong_table = show_correct_wrong_by_class(
    best_results_df,
    best_model_name
)

print("\nBest Model Overall Result:")
print("Total test images:", len(best_results_df))
print("Correct predictions:", int(best_results_df["Correct"].sum()))
print("Wrong predictions:", int((~best_results_df["Correct"]).sum()))
print("Accuracy from result table:", round(best_results_df["Correct"].mean(), 4))

------------------
## Part 5: Single Image Prediction Function
--------------------

#### Prediction Function for Best Model

In [ ]:
def predict_banana_ripeness(image_path, model=None):
    if model is None:
        model = best_model

    image = keras.utils.load_img(image_path, target_size=(IMG_SIZE, IMG_SIZE))
    image_array = keras.utils.img_to_array(image)
    image_array = np.expand_dims(image_array, axis=0)

    prediction = model.predict(image_array, verbose=0)[0]

    predicted_index = int(np.argmax(prediction))
    predicted_class = index_to_class[predicted_index]
    confidence = float(np.max(prediction))

    all_probabilities = {
        index_to_class[i]: float(prediction[i])
        for i in range(len(class_names))
    }

    return {
        "predicted_class": predicted_class,
        "confidence": confidence,
        "all_probabilities": all_probabilities
    }


sample_image = X_test[0]
prediction_result = predict_banana_ripeness(sample_image)

print("Sample image:", sample_image)
print("Prediction result:")
print(json.dumps(prediction_result, indent=4))

img = keras.utils.load_img(sample_image, target_size=(160, 160))
plt.figure(figsize=(4, 4))
plt.imshow(img)
plt.title(
    f"Predicted: {prediction_result['predicted_class']}\n"
    f"Confidence: {prediction_result['confidence']:.2%}"
)
plt.axis("off")
plt.show()

### Recommendation Logic

In [ ]:
def get_recommendation(predicted_class):
    class_name = predicted_class.lower()

    if "unripe" in class_name or "green" in class_name:
        return "The banana may not be ready for immediate consumption. It is better for storage or later use."

    if "ripe" in class_name and "over" not in class_name:
        return "The banana appears suitable for immediate consumption or selling."

    if "overripe" in class_name or "over ripe" in class_name:
        return "The banana should be consumed soon. It may be suitable for smoothies, baking, or processing."

    if "rotten" in class_name or "bad" in class_name or "decay" in class_name:
        return "The banana may no longer be suitable for consumption."

    return "The banana ripeness stage was classified. Please inspect the image and confidence score before making a final decision."


recommendation = get_recommendation(prediction_result["predicted_class"])

print("Predicted class:", prediction_result["predicted_class"])
print("Confidence:", f"{prediction_result['confidence']:.2%}")
print("Recommendation:", recommendation)

In [ ]:
----------------
